#Question and Answering system using RNN in pytorch

## Steps:

1. Loading the dataset
2. Tokenization
3. Vocabulary
4. Convert words to numerical

In [1]:
import pandas as pd
df = pd.read_csv("100_Unique_QA_Dataset.csv")

In [2]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'",'')
  return text.split()

In [4]:
# creating a vocab

vocab = {'<UNK>':0}

In [11]:
def make_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])
  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab) # this is the length of the dict that we are building. it's dynamic.

In [12]:
df.apply(make_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [13]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [14]:
len(vocab)

324

In [15]:
# convert words to numerical indices

def text_to_indices(text, vocab):

  indexed_text = []

  for token in tokenize(text):

    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text

In [16]:
text_to_indices("Who is vraj patel", vocab)

[10, 2, 0, 0]

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader

In [20]:
class QADataset(Dataset):

  def __init__(self, df, vocab):

    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):

    indexed_question = text_to_indices(self.df.iloc[index]['question'], self.vocab)
    indexed_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)

    return torch.tensor(indexed_question), torch.tensor(indexed_answer)


In [21]:
dataset = QADataset(df, vocab)

In [22]:
dataset[32]

(tensor([ 78,  79, 129,  81,  19,   3,  21,  22]), tensor([36]))

In [24]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True) # sice data is small, we have 1 per batch so no need for padding in data either

In [25]:
for question, answer in dataloader:
  print(question, answer)

tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([[36]])
tensor([[  1,   2,   3, 221,   5, 222, 223, 224]]) tensor([[225]])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([[114]])
tensor([[  1,   2,   3,  17, 115,  83,  84]]) tensor([[116]])
tensor([[ 42, 250, 251, 118, 252, 253]]) tensor([[254]])
tensor([[ 42, 137,   2, 138,  39, 175, 269]]) tensor([[99]])
tensor([[ 42, 318,   2,  62,  63,   3, 319,   5, 320]]) tensor([[321]])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([[173]])
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]]) tensor([[295]])
tensor([[ 42, 263, 264,  14, 265, 266, 158, 267]]) tensor([[268]])
tensor([[ 42,  18,   2,   3, 281,  12,   3, 282]]) tensor([[205]])
tensor([[  1,   2,   3,   4,   5, 109]]) tensor([[317]])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([[74]])
tensor([[  1,   2,   3,   4,   5, 135]]) tensor([[136]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]])
tensor([[  1,   2,   3,   4,   5, 206]]) tensor([[207]])
tensor([[  1,   2,   

In [26]:
import torch.nn as nn

In [39]:
class VrajRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50) # embedding_dim is hyperparameter
    self.rnn = nn.RNN(50, 64, batch_first=True)  # need batch_first in rnn's -> this is to get the dims as [1,1,vocab_size] cuz it's now output layer time
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0)) # this is to remove one extra dim

    return output

The reason we did not use sequential layer in the above cell is because of the way rnn works internally.


by default, sequential is designed in a way, that it sends the output of one layer to the very next layer and so and so. and sequential only allows each layer to output one output.


but in rnn, it also output's the intermediate states along with the final output. because it works on timesteps, if there are 5 works in the data, it will output 6 different tensors which have 5 of those intermediate stps and the 6th one is the final  one. so the intermediate ones are the output of hidden states while the final last one is the final output that we are looking for and which will be used by the next layer

so since the output from the nn.RNN will have a total of 6 (for example) tensors, we will only send the last one. but for this, we will have to write it manually. so we will modify the forward() functino

In [44]:
learning_rate = 0.001
epochs=50

In [45]:
model = VrajRNN(len(vocab))

In [46]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [49]:
# training loop
for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    # clearing gradients
    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss
    loss = criterion(output, answer[0])

    # calculate gradients
    loss.backward()

    # update gradients
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch {epoch+1}, Loss: {total_loss:4f}")

Epoch 1, Loss: 0.077944
Epoch 2, Loss: 0.074514
Epoch 3, Loss: 0.071115
Epoch 4, Loss: 0.067952
Epoch 5, Loss: 0.064906
Epoch 6, Loss: 0.062113
Epoch 7, Loss: 0.059227
Epoch 8, Loss: 0.056564
Epoch 9, Loss: 0.054066
Epoch 10, Loss: 0.051645
Epoch 11, Loss: 0.049349
Epoch 12, Loss: 0.047113
Epoch 13, Loss: 0.045019
Epoch 14, Loss: 0.043022
Epoch 15, Loss: 0.041110
Epoch 16, Loss: 0.039308
Epoch 17, Loss: 0.037580
Epoch 18, Loss: 0.035884
Epoch 19, Loss: 0.034330
Epoch 20, Loss: 0.032796
Epoch 21, Loss: 0.031340
Epoch 22, Loss: 0.029961
Epoch 23, Loss: 0.028662
Epoch 24, Loss: 0.027361
Epoch 25, Loss: 0.026157
Epoch 26, Loss: 0.025018
Epoch 27, Loss: 0.023908
Epoch 28, Loss: 0.022857
Epoch 29, Loss: 0.021845
Epoch 30, Loss: 0.020892
Epoch 31, Loss: 0.019986
Epoch 32, Loss: 0.019076
Epoch 33, Loss: 0.018254
Epoch 34, Loss: 0.017446
Epoch 35, Loss: 0.016680
Epoch 36, Loss: 0.015958
Epoch 37, Loss: 0.015253
Epoch 38, Loss: 0.014598
Epoch 39, Loss: 0.013942
Epoch 40, Loss: 0.013338
Epoch 41,

In [52]:
def predict_answer(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max probability
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("idk bruvv :*(")
  else:
    print(list(vocab.keys())[index])

In [54]:
predict_answer(model, "What is the capital of india?")

delhi


In [55]:
predict_answer(model, "What is delhi?")

idk bruvv :*(
